In [1]:
from utils.datasettemporal import TemporalDataset

In [ ]:
dataset = TemporalDataset(filepath="data/daily_alternative_small/small_daily_alternative_sample_1993-1993.nc")
print(len(dataset))
dataset_loaded = [dataset[i] for i in range(500000)]

In [7]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

model = HistGradientBoostingRegressor()
groups = dataset.groups
all_indices = list(range(len(dataset)))

gss = GroupKFold(n_splits=5, random_state=42, shuffle=True)
gss_indices = list(gss.split(all_indices, groups=groups))
for train_indices, test_indices in gss_indices:
    for i in train_indices:
        X, y = dataset[i]
        model.partial_fit(X, y)
    for i in test_indices:
        X, y = dataset[i]
        preds = model.predict(X)
        y_true = []
        y_pred = []
        y_true.extend(y.ravel())
        y_pred.extend(preds.ravel())
    mse = mean_squared_error(y_true, y_pred)
    print(f"Fold MSE: {mse}")

AttributeError: 'HistGradientBoostingRegressor' object has no attribute 'partial_fit'

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import GroupKFold

model = HistGradientBoostingRegressor()
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200]
}

groups = dataset.groups
all_indices = list(range(len(dataset)))

gss_test = GroupKFold(n_splits=5, random_state=42, shuffle=True)
grid_search = GridSearchCV(model, param_grid, cv=gss_test, scoring='neg_mean_squared_error')
X = dataset[0][0].reshape(dataset[0][0].shape[0], -1)  # Flatten the input data
y = dataset[0][1].ravel()  # Flatten the target data
groups = dataset.groups
